# Sampling and post-processing of data

Actions:
- Read simulation output in its different formats
- Adaptation to ML-compatible structures (matrix, implicit representations...)
- Save ready for usage

In [ ]:
import json
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay, KDTree
from scipy.interpolate import LinearNDInterpolator

In [ ]:
DATASET_INPUT = "cylinder"
DATASET_NAME = "cylinder96"
MAX_RESOLUTION = 96 // 16 * 16  # Always multiples of 16
SUPER_RESOLUTION_FACTOR = 30
CROP_X = [2, 18]
CROP_Y = [1.8, 8.2]

dataset_path = Path.cwd().parent / f"data/01_raw/{DATASET_INPUT}"

output_path = Path.cwd().parent / f"data/02_sampled/{DATASET_NAME}"

In [ ]:
import matplotlib.path as mpltPath

Path(output_path).mkdir(parents=True, exist_ok=True)
cases = sorted([file for file in dataset_path.iterdir() if file.suffix == ".h5" and file.stem != "constants"], key=lambda x: float(x.stem))

# Mesh rasterization
with h5py.File(dataset_path / "constants.h5", "r") as file:
    mesh_centers = file["centroids"][:]
    mesh_indices = file["indices"][:]
    mesh_vertices = file["vertices"][:]

N_cells = len(mesh_centers)
mesh_min, mesh_max = mesh_vertices.min(axis=0), mesh_vertices.max(axis=0)
mesh_range = mesh_max - mesh_min
if CROP_X is None: CROP_X = [mesh_min[0], mesh_max[0]]
if CROP_Y is None: CROP_Y = [mesh_min[1], mesh_max[1]]

# Determine the resolution for each dimension based on the maximum resolution and the aspect ratio of the mesh
max_dim = np.argmax(mesh_range)
res = (MAX_RESOLUTION * mesh_range / mesh_range[max_dim]).astype(int) // 16 * 16  # Round down to the nearest multiple of 16 for better downsampling later

super_res = (res[0]*SUPER_RESOLUTION_FACTOR, res[1]*SUPER_RESOLUTION_FACTOR)
super_res_grid = -1*np.ones(super_res, dtype=int)

mesh_vertices_SR = (mesh_vertices - mesh_min) / mesh_range * (np.array(super_res)+0.0001)

for centroid in range(N_cells):
    vertices = mesh_vertices_SR[mesh_indices[centroid]]
    bounding_box_max, bounding_box_min = np.max(np.ceil(vertices), axis=0).astype(int), np.min(np.floor(vertices), axis=0).astype(int)

    points = np.mgrid[bounding_box_min[0]:bounding_box_max[0]+1, bounding_box_min[1]:bounding_box_max[1]+1].reshape(2, -1).T

    mask = mpltPath.Path(vertices).contains_points(points+0.5)
    super_res_grid[points[mask][:,0], points[mask][:,1]] = centroid

# show an image using super_res_grid
plt.figure(figsize=(8,8))
plt.imshow(super_res_grid.T, origin="lower")
plt.title("Super-resolution grid")
plt.show()

In [ ]:
# Crop
crop_super_res_x = np.floor((CROP_X - mesh_min[0]) / mesh_range[0] * super_res[0]).astype(int)
crop_super_res_x = crop_super_res_x // res[0] * res[0]
crop_super_res_y = np.floor((CROP_Y - mesh_min[1]) / mesh_range[1] * super_res[1]).astype(int)
crop_super_res_y = crop_super_res_y // res[1] * res[1]
super_res_grid = super_res_grid[crop_super_res_x[0]:crop_super_res_x[1], crop_super_res_y[0]:crop_super_res_y[1]]

super_res = super_res_grid.shape
# What cells in the grid are valid
super_res_grid_mask = super_res_grid != -1
# Masked super-resolution grid
super_res_grid_masked = super_res_grid[super_res_grid_mask]

# show an image using super_res_grid
plt.figure(figsize=(8,8))
plt.imshow(super_res_grid.T, origin="lower")
plt.title("Super-resolution grid")
plt.show()

In [ ]:
masked_field = super_res_grid.copy().astype(float)
masked_field[~super_res_grid_mask] = np.nan
downsampled_grid = np.nanmean(masked_field.reshape(
    res[0], super_res[0]//res[0],
    res[1], super_res[1]//res[1]
), axis=(1,3))

# show an image using downsampled_grid
plt.figure(figsize=(8,8))
plt.imshow(downsampled_grid.T, origin="lower")
plt.title("Downsampled grid")
plt.show()

In [ ]:
# Validity
nan_counts = np.isnan(masked_field).reshape(
    res[0], super_res[0]//res[0],
    res[1], super_res[1]//res[1]
).sum(axis=(1,3))
points_per_block = (super_res[0]//res[0]) * (super_res[1]//res[1])
validity = 1 - nan_counts / points_per_block

plt.figure(figsize=(8,8))
plt.imshow(validity.T, origin="lower")
plt.title("Validity")
plt.show()

In [ ]:
# SDF
from scipy.ndimage import distance_transform_edt # Exact Euclidean distance transform
from matplotlib.colors import TwoSlopeNorm

outside = distance_transform_edt(super_res_grid_mask == 0)
inside  = distance_transform_edt(super_res_grid_mask == 1)
sdf = outside - inside

sdf = np.mean(sdf.reshape(
    res[0], super_res[0]//res[0],
    res[1], super_res[1]//res[1]
), axis=(1,3))

plt.figure(figsize=(8,8))
if sdf.max() >= 0:
    norm = TwoSlopeNorm(vmin=sdf.min(), vcenter=0, vmax=sdf.max())
else: 
    sdf = np.zeros_like(sdf)
    norm = None
plt.imshow(sdf.T, origin="lower", cmap="coolwarm", norm=norm)
plt.title("Signed distance field")
plt.show()

In [ ]:
# What cells in the grid are invalid
res_nan_mask = np.isnan(downsampled_grid)

In [ ]:
import json

### Dataset constants
with open(dataset_path / "constants.json", "r") as file:
    constants = json.load(file)

constants["source_dataset"] = DATASET_INPUT
constants["fields"] = ["p", "Ux", "Uy"]
with open(output_path / "constants.json", "w") as file:
    json.dump(constants, file, indent=4)

with h5py.File(output_path / "constants.h5", "w") as file:
    file.create_dataset("validity", data=validity)
    file.create_dataset("sdf", data=sdf)

In [ ]:
def process_case(case_path):
    super_res_field = np.full(super_res, np.nan, dtype=float)
    with h5py.File(case_path, "r") as file:
        attrs = dict(file.attrs)
        time = file["t"][:]
        data = {
            "p": file["p"][:],
            "Ux": file["U"][:,:,0],
            "Uy": file["U"][:,:,1],
        }
    N_snaps = time.shape[0]

    with h5py.File(output_path / case_path.name, "w") as file:
        file.attrs.update(attrs)
        file.create_dataset("t", data=time, compression="gzip")

        N_keys = len(data)
        new_data = np.zeros((N_keys, N_snaps, res[0], res[1]), dtype=float)
        for i, key in enumerate(data):
            for snapshot in range(N_snaps):
                field = data[key][snapshot]
                
                # gradient = gradient_func(field)
                # super_res_field[super_res_grid_mask] = field[super_res_grid_masked] + (gradient[:,super_res_grid_masked].T * relative_coords).sum(axis=1)
                super_res_field[super_res_grid_mask] = field[super_res_grid_masked]
                
                new_data[i][snapshot] = np.nanmean(super_res_field.reshape(
                    res[0], super_res[0]//res[0],
                    res[1], super_res[1]//res[1]
                ), axis=(1,3))

        file.create_dataset("fields", data=new_data, compression="gzip", chunks=(N_keys, N_snaps,1,1))

from joblib import Parallel, delayed
result = Parallel(n_jobs=-1)(delayed(process_case)(case_path) for case_path in tqdm(cases))
# process_case(cases[-1])